# Fine-tune DistilBERT for Seven-Emotion YouTube Comment Analysis

This notebook trains in two stages:
1. Fine-tune `distilbert-base-uncased` on balanced GoEmotions seven-class data.
2. Continue training on **balanced** YouTube-domain data (neutral downsampled from 1226 to 200).

Labels: `anger`, `disgust`, `fear`, `joy`, `neutral`, `sadness`, `surprise`.

Recommended runtime: **T4 GPU**.


## 1. Install dependencies


In [ ]:
!pip install -q "transformers>=4.46" datasets accelerate evaluate scikit-learn pandas pyarrow huggingface_hub


## 2. Imports and label setup


In [ ]:
import inspect
import random
import time

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments, pipeline

import datasets.config as datasets_config
datasets_config.TORCHVISION_AVAILABLE = False

SEED = 5240
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

TARGET_LABELS = ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"]
LABEL_TO_ID = {label: i for i, label in enumerate(TARGET_LABELS)}
ID_TO_LABEL = {i: label for label, i in LABEL_TO_ID.items()}

print("Labels:", TARGET_LABELS)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 3. Helper functions


In [ ]:
def make_training_args(**kwargs):
    sig = inspect.signature(TrainingArguments.__init__).parameters
    if "eval_strategy" in sig:
        kwargs["eval_strategy"] = kwargs.pop("evaluation_strategy", "epoch")
    else:
        kwargs["evaluation_strategy"] = kwargs.pop("evaluation_strategy", "epoch")
    return TrainingArguments(**kwargs)


def make_trainer(model, args, train_dataset, eval_dataset, tokenizer, compute_metrics, class_weights=None):
    kwargs = {
        "model": model,
        "args": args,
        "train_dataset": train_dataset,
        "eval_dataset": eval_dataset,
        "compute_metrics": compute_metrics,
    }
    sig = inspect.signature(Trainer.__init__).parameters
    if "processing_class" in sig:
        kwargs["processing_class"] = tokenizer
    elif "tokenizer" in sig:
        kwargs["tokenizer"] = tokenizer

    if class_weights is not None:
        class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)
        if torch.cuda.is_available():
            class_weights_tensor = class_weights_tensor.cuda()

        class WeightedTrainer(Trainer):
            def compute_loss(self, model, inputs, return_outputs=False, **kw):
                labels = inputs.pop("labels")
                outputs = model(**inputs)
                logits = outputs.logits
                loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights_tensor)
                loss = loss_fn(logits, labels)
                return (loss, outputs) if return_outputs else loss

        return WeightedTrainer(**kwargs)

    return Trainer(**kwargs)


def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    wp, wr, wf1, _ = precision_recall_fscore_support(labels, preds, average="weighted", zero_division=0)
    mp, mr, mf1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    return {"accuracy": acc, "weighted_f1": wf1, "macro_f1": mf1,
            "weighted_precision": wp, "weighted_recall": wr,
            "macro_precision": mp, "macro_recall": mr}


## 4. Load GoEmotions dataset

Keep single-label samples for our 7 target emotions, then balance by downsampling.


In [ ]:
PARQUET_URLS = {
    "train": "https://huggingface.co/datasets/SetFit/go_emotions/resolve/refs%2Fconvert%2Fparquet/default/train/0000.parquet",
    "validation": "https://huggingface.co/datasets/SetFit/go_emotions/resolve/refs%2Fconvert%2Fparquet/default/validation/0000.parquet",
    "test": "https://huggingface.co/datasets/SetFit/go_emotions/resolve/refs%2Fconvert%2Fparquet/default/test/0000.parquet",
}

def filter_single_target(df):
    label_cols = [c for c in df.columns if c != "text"]
    single = df[df[label_cols].sum(axis=1) == 1].copy()
    target = single[single[TARGET_LABELS].sum(axis=1) == 1].copy()
    target["label_name"] = target[TARGET_LABELS].idxmax(axis=1)
    target["label"] = target["label_name"].map(LABEL_TO_ID).astype(int)
    return target[["text", "label_name", "label"]].reset_index(drop=True)

def balance_by_label(df, rs=SEED):
    mn = df["label_name"].value_counts().min()
    parts = [df[df["label_name"] == l].sample(n=mn, random_state=rs) for l in sorted(df["label_name"].unique())]
    return pd.concat(parts, ignore_index=True).sample(frac=1.0, random_state=rs).reset_index(drop=True)

go_frames = {}
for split, url in PARQUET_URLS.items():
    raw = pd.read_parquet(url)
    filtered = filter_single_target(raw)
    go_frames[split] = filtered
    print(split, go_frames[split].shape)
    print(go_frames[split]["label_name"].value_counts().sort_index())


## 5. Convert to HF Dataset


In [ ]:
go_dataset = DatasetDict({
    s: Dataset.from_pandas(df[["text", "label"]], preserve_index=False)
    for s, df in go_frames.items()
})
go_dataset


## 6. Tokenize


In [ ]:
BASE_MODEL = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

go_tokenized = go_dataset.map(tokenize_batch, batched=True)
go_tokenized = go_tokenized.remove_columns(["text"])
go_tokenized


## 7. Load model


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=len(TARGET_LABELS), id2label=ID_TO_LABEL, label2id=LABEL_TO_ID,
)


## 8. Stage 1: GoEmotions training


In [ ]:
s1_args = make_training_args(
    output_dir="./s1-goemotions-results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=50,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
)

s1_trainer = make_trainer(model, s1_args, go_tokenized["train"], go_tokenized["validation"], tokenizer, compute_metrics)

t0 = time.time()
s1_trainer.train()
print("Stage 1 seconds:", round(time.time() - t0, 2))


## 9. Evaluate Stage 1


In [ ]:
s1_val = s1_trainer.evaluate(go_tokenized["validation"])
s1_test = s1_trainer.evaluate(go_tokenized["test"])
print("Stage 1 validation:", s1_val)
print("Stage 1 test:", s1_test)


## 10. Save Stage 1 model


In [ ]:
S1_DIR = "./youtube-emotion-distilbert"
s1_trainer.save_model(S1_DIR)
tokenizer.save_pretrained(S1_DIR)
print("Saved to", S1_DIR)

samples = [
    "I love this video so much!",
    "This campaign makes me angry.",
    "I am shocked by this announcement.",
    "This is just a normal update.",
]
pipe1 = pipeline("text-classification", model=S1_DIR, tokenizer=S1_DIR)
pipe1(samples, truncation=True, return_token_type_ids=False)


## 11. Load balanced YouTube-domain data

The balanced dataset has neutral downsampled from 1226 to 200 in training.
This forces the model to learn finer distinctions instead of defaulting to neutral.


In [ ]:
YOUTUBE_BASE = "https://raw.githubusercontent.com/chasezhang1999/youtube-emotion-analyzer/main/data/youtube_domain_7class_balanced"

yt_train = pd.read_csv(f"{YOUTUBE_BASE}/train.csv")
yt_val = pd.read_csv(f"{YOUTUBE_BASE}/validation.csv")

for df in [yt_train, yt_val]:
    df["text"] = df["text"].astype(str)
    df["label_name"] = df["label"].astype(str)
    df["label"] = df["label_id"].astype(int)

print("YouTube-domain balanced train:", yt_train.shape)
print(yt_train["label_name"].value_counts().sort_index())
print("\nYouTube-domain balanced validation:", yt_val.shape)
print(yt_val["label_name"].value_counts().sort_index())


## 12. Tokenize YouTube-domain data


In [ ]:
yt_dataset = DatasetDict({
    "train": Dataset.from_pandas(yt_train[["text", "label"]], preserve_index=False),
    "validation": Dataset.from_pandas(yt_val[["text", "label"]], preserve_index=False),
})
yt_tokenized = yt_dataset.map(tokenize_batch, batched=True)
yt_tokenized = yt_tokenized.remove_columns(["text"])
yt_tokenized


## 13. Compute class weights for domain adaptation

Because even the balanced data has some imbalance, we compute inverse-frequency weights.
This penalizes the model more for misclassifying minority classes.


In [ ]:
train_label_counts = yt_train["label"].value_counts().sort_index().values.astype(float)
class_weights = train_label_counts.sum() / (len(TARGET_LABELS) * train_label_counts)
class_weights = class_weights / class_weights.min()  # normalize so min weight = 1
print("Class weights:")
for label, w, c in zip(TARGET_LABELS, class_weights, train_label_counts):
    print(f"  {label}: {w:.2f}  (n={int(c)})")


## 14. Stage 2: YouTube-domain adaptation with class weights

Using `macro_f1` for best-model selection and class-weighted loss to reduce neutral bias.


In [ ]:
domain_model = AutoModelForSequenceClassification.from_pretrained(
    S1_DIR, num_labels=len(TARGET_LABELS), id2label=ID_TO_LABEL, label2id=LABEL_TO_ID,
)

s2_args = make_training_args(
    output_dir="./s2-domain-results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=20,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
)

s2_trainer = make_trainer(
    domain_model, s2_args,
    yt_tokenized["train"], yt_tokenized["validation"],
    tokenizer, compute_metrics,
    class_weights=class_weights.tolist(),
)

t0 = time.time()
s2_trainer.train()
print("Stage 2 seconds:", round(time.time() - t0, 2))


## 15. Evaluate Stage 2


In [ ]:
s2_val = s2_trainer.evaluate(yt_tokenized["validation"])
print("Stage 2 YouTube-domain validation:", s2_val)


## 16. Save domain-adapted model


In [ ]:
S2_DIR = "./youtube-emotion-distilbert-domain-adapted"
s2_trainer.save_model(S2_DIR)
tokenizer.save_pretrained(S2_DIR)
print("Saved to", S2_DIR)

pipe2 = pipeline("text-classification", model=S2_DIR, tokenizer=S2_DIR)
pipe2(samples, truncation=True, return_token_type_ids=False)


## 17. Compare Stage 1 vs Stage 2


In [ ]:
comparison = [
    "This launch is amazing and I want to buy it now!",
    "The brand response is terrible and people are angry.",
    "This safety ad is scary but important.",
    "I did not expect that ending at all.",
    "Just here to check the product details.",
    "This company should be ashamed of themselves.",
    "Watching from Ghana, love this!",
    "This made me cry so much.",
]

p1 = pipeline("text-classification", model=S1_DIR, tokenizer=S1_DIR)
p2 = pipeline("text-classification", model=S2_DIR, tokenizer=S2_DIR)

rows = []
for t in comparison:
    r1 = p1(t, truncation=True, return_token_type_ids=False)[0]
    r2 = p2(t, truncation=True, return_token_type_ids=False)[0]
    rows.append({"text": t, "stage1": r1["label"], "s1_conf": round(r1["score"], 3),
                 "domain": r2["label"], "dom_conf": round(r2["score"], 3)})
pd.DataFrame(rows)


## 18. Upload to Hugging Face

Add your HF write token to Colab Secrets as `HF_TOKEN` before running.


In [ ]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("HF_TOKEN")
login(token=hf_token)

repo1 = "chase1zhang/youtube-emotion-distilbert"
repo2 = "chase1zhang/youtube-emotion-distilbert-domain-adapted"

s1_trainer.model.push_to_hub(repo1)
tokenizer.push_to_hub(repo1)
print(f"Uploaded Stage 1: https://huggingface.co/{repo1}")

s2_trainer.model.push_to_hub(repo2)
tokenizer.push_to_hub(repo2)
print(f"Uploaded domain-adapted: https://huggingface.co/{repo2}")


## 19. Verify uploaded models


In [ ]:
v1 = pipeline("text-classification", model=repo1, tokenizer=repo1)
v2 = pipeline("text-classification", model=repo2, tokenizer=repo2)
print("Stage 1:", v1(samples, truncation=True, return_token_type_ids=False))
print("Domain:", v2(samples, truncation=True, return_token_type_ids=False))


## 20. Summary

Copy these into the report:
- Stage 1 GoEmotions validation / test metrics
- Stage 2 YouTube-domain validation metrics
- Hugging Face model URLs

The Streamlit app default model: `chase1zhang/youtube-emotion-distilbert-domain-adapted`
